In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
from dotenv import dotenv_values
env = dotenv_values(".env")

In [3]:
engine = create_engine(env["POSTGRES_URI"], pool_pre_ping=True,pool_recycle=900,pool_timeout=30)

def check_postgres_connection():
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 'Hello, World!'"))
        for row in result:
            print(row[0])

In [4]:
def check_table_exists(table_name):
    with engine.connect() as connection:
        result = connection.execute(text(f"SELECT EXISTS (SELECT FROM information_schema.tables WHERE table_name = '{table_name}');"))
        exists = result.scalar()
        return exists

In [5]:
def add_table_to_postgres(df, table_name):
    if check_table_exists(table_name):
        print(f"Table '{table_name}' already exists in the database.")
    else:
        df.to_sql(table_name, engine, index=False)
        print(f"Table '{table_name}' has been added to the database.")

In [6]:
customers = pd.read_csv("./csv_source_data/customers.csv")
orders = pd.read_csv("./csv_source_data/orders.csv")

In [7]:
customers.head()

,customer_id,name,city,segment,signup_date
0,1,Usman Sheikh,Karachi,Consumer,2022-11-28
1,2,Iqra Khan,Faisalabad,Consumer,2022-11-11
2,3,Maryam Raza,Faisalabad,Consumer,2023-10-20
3,4,Ali Sheikh,Lahore,Consumer,2022-08-09
4,5,Hamza Qureshi,Karachi,Consumer,2022-11-10


In [8]:
orders.head()

,order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at
0,256821,1419,2025-01-01,Beauty,1,575.98,pending,wallet,2025-01-01 05:34:09
1,208443,701,2025-01-01,Fashion,2,5685.46,delivered,bank_transfer,2025-01-05 08:11:23
2,265974,430,2025-01-01,Electronics,1,35507.97,delivered,wallet,2025-01-03 06:08:32
3,221969,307,2025-01-01,Beauty,1,1383.10,delivered,bank_transfer,2025-01-08 06:29:24
4,271915,541,2025-01-01,Home,3,69759.11,delivered,wallet,2025-01-03 07:45:02


In [14]:
add_table_to_postgres(customers, "dataengineering_project_customers")
add_table_to_postgres(orders, "dataengineering_project_orders")

Table 'dataengineering_project_customers' already exists in the database.
Table 'dataengineering_project_orders' has been added to the database.


In [17]:
from pymongo import MongoClient, ASCENDING
from pymongo.errors import BulkWriteError
import certifi 

In [11]:
MONGO_URI = env["MONGO_URI"]

In [20]:
def initialize_mongodb_structure(db_name, collection_name):
    client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
    
    try:
        db = client[db_name]
        
        if collection_name in db.list_collection_names():
            print(f"Collection '{collection_name}' already exists in database '{db_name}'.")
        else:
            db.create_collection(collection_name)
            print(f"Successfully created collection structure: '{collection_name}'")
            
        collection = db[collection_name]
        
        index_name = collection.create_index([("order_id", ASCENDING)], unique=True)
        print(f"Created structural index target: {index_name}")
        
    except Exception as e:
        print(f"Failed to build MongoDB structure: {e}")
        
    finally:
        client.close()

initialize_mongodb_structure("dataengineering_db", "orders")


Successfully created collection structure: 'orders'
Created structural index target: order_id_1


In [23]:
def ingest_orders_json_with_pandas(json_file_path, db_name="dataengineering_db", collection_name="orders"):
    df = pd.read_json(json_file_path)
    df = df.where(pd.notnull(df), None)
    orders_batch = df.to_dict(orient="records")
    client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
    
    try:
        db = client[db_name]
        collection = db[collection_name]
        
        if orders_batch:
            print(f"Streaming {len(orders_batch)} objects to MongoDB Atlas...")
            result = collection.insert_many(orders_batch, ordered=False)
            print(f"Success! Loaded {len(result.inserted_ids)} new documents into '{collection_name}'.")
        else:
            print("Dataframe returned empty. Operation halted.")
            
    except BulkWriteError as bwe:
        inserted_count = bwe.details.get("nInserted", 0)
        print(f"Inserted {inserted_count} new unique records, skipped duplicates.")
    except Exception as e:
        print(f"Failed to stream records: {e}")
    finally:
        client.close()

json_filename = "./csv_source_data/orders.json" 
ingest_orders_json_with_pandas(json_filename)

Streaming 5000 objects to MongoDB Atlas...
Success! Loaded 5000 new documents into 'orders'.
